[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-07-toy-model.ipynb)

# Lecture 7: your first model, and what it does not know

This is one exercise in three parts.

| part | where | what you get |
|---|---|---|
| **A. In class** | twenty minutes | a working install, a trained toy model, a loss curve you can read, and Google Drive connected for the A8 notebook |
| **B. Outside class** | on your own | the same model with a phrase of your choice held out, a second phrase as the contrast, two scores that disagree, and a training run that survives a disconnect |
| **C. Turn in** | on Learning Suite | your prediction, the printed report, and two sentences |

The point of Part B is the point of Lecture 8a: a model that has memorised its training data can
score perfectly on it and still know nothing. You will see that happen on twelve phrases, and
then in Lecture 8a you will hear why it happens on 100,000.

**In Colab, set Runtime → Change runtime type → GPU first.** Today's model trains on a CPU in
under a minute; the A8 model will not, so build the habit now.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**CS 479 course notebook** — the in-class activity for Lecture 7 (*Research Paper Reviews; Intro to Neural Networks*). A head start on **A7**.

## Part A, in class

In [ ]:
# Install TorchLingo in Colab. Nothing to uncomment. Pip's "you may need to restart the
# kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo>=0.2.2"
import torchlingo

In [ ]:
# Check the setup. If this cell raises, stop here and put your hand up.
from torchlingo.colab import setup

env = setup()                  # prints the versions and the device
device = env.device            # a GPU is fine to lack today; not in Lecture 8a

import torch
from pathlib import Path
from torch.utils.data import DataLoader
from torchlingo.config import Config
from torchlingo.data_processing import NMTDataset, collate_fn
from torchlingo.models import SimpleTransformer
from torchlingo.training import train_model
from torchlingo.training_checkpoint import TrainingCheckpointer
from torchlingo.inference import translate_batch
from torchlingo.diagnostics import uniform_loss

### A1. The corpus

Twelve English phrases and their Spanish, each repeated sixteen times so the model has enough
practice to memorise them. Notice the last four: **the cat sleeps, the dog runs, the cat runs,
the dog sleeps** share every word between them. Hold onto that; it is what Part B is about.

In [ ]:
import pandas as pd

BASE_PAIRS = [
    ("Hello world", "Hola mundo"),
    ("Good morning", "Buenos días"),
    ("Good night", "Buenas noches"),
    ("Thank you", "Gracias"),
    ("I love you", "Te amo"),
    ("How are you", "Cómo estás"),
    ("See you tomorrow", "Hasta mañana"),
    ("The cat sleeps", "El gato duerme"),
    ("The dog runs", "El perro corre"),
    ("The cat runs", "El gato corre"),
    ("The dog sleeps", "El perro duerme"),
    ("I love the cat", "Amo al gato"),
]
REPEATS = 16

def write_corpus(pairs, path):
    df = pd.DataFrame(pairs * REPEATS, columns=["src", "tgt"])
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, sep="\t", index=False)
    return df

df = write_corpus(BASE_PAIRS, "data/train_all.tsv")
print(f"{len(df)} training examples from {len(BASE_PAIRS)} distinct phrases.")

### A2. Vocabulary, model, and the number to compare against

Three things the slides told you to look at get printed here: the vocabulary size, the parameter
count, and `ln(V)`, the loss of a model that has learned nothing and guesses every token equally.

In [ ]:
config = Config(
    d_model=64, n_heads=4,
    num_encoder_layers=2, num_decoder_layers=2,
    d_ff=128, dropout=0.1,
    batch_size=8, learning_rate=1e-3,
    num_workers=0,
)

def build(train_tsv, seed=0):
    ds = NMTDataset(train_tsv, config=config)
    loader = DataLoader(ds, batch_size=config.batch_size, shuffle=True, collate_fn=collate_fn, num_workers=0)
    torch.manual_seed(seed)
    model = SimpleTransformer(
        src_vocab_size=len(ds.src_vocab), tgt_vocab_size=len(ds.tgt_vocab), config=config,
    ).to(device)
    return ds, loader, model

train_ds, train_loader, model = build("data/train_all.tsv")
V = len(train_ds.tgt_vocab)
n_params = sum(p.numel() for p in model.parameters())
print(f"Source vocabulary: {len(train_ds.src_vocab)} tokens.  Target vocabulary V = {V} tokens.")
print(f"Parameters: {n_params:,}")
print(f"ln(V) = {uniform_loss(V):.3f}   <- a first-batch loss near this means 'not learned anything yet'")

### A3. Train

Forty epochs: seconds on a GPU, under a minute on a CPU. Compare the **first** logged loss with
`ln(V)` above. Expect it a little *above* `ln(V)`, because a fresh model's random weights
already lean the wrong way on some tokens; what shows learning is the curve crossing `ln(V)`
and continuing down.

In [ ]:
EPOCHS = 40
result = train_model(
    model, train_loader,
    num_epochs=EPOCHS, device=device, config=config,
    gradient_clip=1.0, log_every=5,
    # Saves progress at every epoch. Running this cell again continues from the save instead of
    # retraining, so a second run finishes at once. Part B shows why that matters.
    checkpointer=TrainingCheckpointer("lecture-07-all", checkpoint_dir="checkpoints/lecture-07-all", verbose=False),
)
losses = result.train_losses
print(f"\nEpoch-1 average loss: {losses[0]:.3f}   final loss: {losses[-1]:.3f}   ln(V): {uniform_loss(V):.3f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 3.5))
plt.plot(range(1, len(losses) + 1), losses, label="training loss (epoch mean)")
plt.axhline(uniform_loss(V), linestyle="--", label=f"ln(V) = {uniform_loss(V):.2f}")
plt.xlabel("epoch"); plt.ylabel("cross-entropy"); plt.grid(alpha=0.3); plt.legend()
plt.title("Your first loss curve")
plt.show()

### A4. Translate the training phrases

In [ ]:
def translate(model, ds, sentences):
    return translate_batch(model, sentences, ds.src_vocab, ds.tgt_vocab,
                           decode_strategy="greedy", device=device, config=config)

for src, hyp in zip([p[0] for p in BASE_PAIRS[:6]], translate(model, train_ds, [p[0] for p in BASE_PAIRS[:6]])):
    print(f"{src:20} -> {hyp}")

### A5. Connect Google Drive now

The A8 notebook reads your Assignment 5 corpus from Drive and saves its checkpoints there,
because a Colab runtime forgets everything when it disconnects. The first mount asks you to
choose an account and approve access; do it now, while someone can help. The next cell creates
`MyDrive/CS479`, the folder the A8 notebook looks in, and writes a test file there.

In [ ]:
# Mount Google Drive and prove you can write to it. The A8 notebook makes this same call.
env = setup(drive=True)        # in Colab, Google asks you to approve access the first time

COURSE_DIR = env.drive_dir / "CS479"
COURSE_DIR.mkdir(parents=True, exist_ok=True)
check = COURSE_DIR / "lecture-07-drive-check.txt"
check.write_text("Drive works. The A8 notebook reads and writes this folder.\n", encoding="utf-8")

print(f"Wrote {check}")
print()
print("Put your two Assignment 5 files in this folder:")
print(f"  {COURSE_DIR}/")
print("The A8 notebook looks for them there; you will type their names into it.")

**Report back, in class:** your first logged loss and `ln(V)`; your final loss; whether the
translations are right; and anything that broke.

---

## Part B, outside class: hold one phrase out

Everything above scored the model on sentences it was trained on. That is not a test; it is a
memory check. So: choose one of the twelve phrases, remove it from the training set, train a
fresh model on the other eleven, and ask it to translate the one it never saw.

Choose with the corpus in front of you. The last four phrases share every word between them, so
a model that holds one of them out has seen every piece of it somewhere else. Most of the others
contain a word that appears nowhere else. Which kind you pick decides what the experiment can
show.

**Before you run anything, write your prediction in the cell below:** will the model produce the
right translation, and why? The report at the end prints it, and says so if it is missing.

In [ ]:
# Choose any pair from BASE_PAIRS. The default is one of the four that share all their words.
HELD_OUT = ("The dog sleeps", "El perro duerme")

# Your prediction, in one sentence: will the model translate HELD_OUT correctly, and why?
PREDICTION = ""

assert HELD_OUT in BASE_PAIRS, "HELD_OUT must be one of the twelve pairs above."
if not PREDICTION.strip():
    print("No prediction written. Write one before you run Part B; the report will say it is missing.\n")

import sacrebleu


def hold_out(pair, label):
    """Train on the other eleven phrases; score the eleven (seen) and the one (unseen)."""
    train_pairs = [p for p in BASE_PAIRS if p != pair]
    assert len(train_pairs) == 11
    write_corpus(train_pairs, f"data/train_{label}.tsv")
    ds, loader, m = build(f"data/train_{label}.tsv")
    # One checkpoint folder per held-out phrase, so a different phrase trains a new model rather
    # than continuing an old one.
    name = "lecture-07-without-" + pair[0].lower().replace(" ", "-")
    train_model(m, loader, num_epochs=EPOCHS, device=device, config=config, gradient_clip=1.0,
                log_every=0, checkpointer=TrainingCheckpointer(name, checkpoint_dir=f"checkpoints/{name}", verbose=False))
    seen_src = [p[0] for p in train_pairs]
    seen_ref = [p[1] for p in train_pairs]
    seen_hyp = translate(m, ds, seen_src)
    unseen_hyp = translate(m, ds, [pair[0]])[0]
    # Exact match is the honest metric for three-word phrases. chrF works on short text too.
    # One hypothesis stream, one list of reference streams.
    return dict(
        pair=pair, unseen_hyp=unseen_hyp,
        exact_seen=sum(h == r for h, r in zip(seen_hyp, seen_ref)),
        chrf_seen=sacrebleu.corpus_chrf(seen_hyp, [seen_ref]).score,
        chrf_unseen=sacrebleu.corpus_chrf([unseen_hyp], [[pair[1]]]).score,
        bleu_seen=sacrebleu.corpus_bleu(seen_hyp, [seen_ref]).score,
        bleu_unseen=sacrebleu.corpus_bleu([unseen_hyp], [[pair[1]]]).score,
    )


def report(r):
    print(f"Held out {r['pair'][0]!r}")
    print(f"  the 11 phrases it trained on:  {r['exact_seen']}/11 exactly right,  chrF {r['chrf_seen']:5.1f}")
    print(f"  the 1 phrase it never saw:     chrF {r['chrf_unseen']:5.1f}   -> {r['unseen_hyp']!r}   (wanted {r['pair'][1]!r})")
    print(f"  BLEU, seen {r['bleu_seen']:.1f}   unseen {r['bleu_unseen']:.1f}")


mine = hold_out(HELD_OUT, "mine")
report(mine)

In [ ]:
# The contrast: a phrase of the other kind. "Good night" shares no word with any other phrase
# (night -> noches appears nowhere else), so there is nothing to recombine from. If the phrase
# you chose is of that kind, the contrast is one of the four that share their words.
SHARED = {("The cat sleeps", "El gato duerme"), ("The dog runs", "El perro corre"),
          ("The cat runs", "El gato corre"), ("The dog sleeps", "El perro duerme")}
CONTRAST = ("Good night", "Buenas noches") if HELD_OUT in SHARED else ("The dog sleeps", "El perro duerme")

contrast = hold_out(CONTRAST, "contrast")
report(contrast)

**What to look at.** Both seen lines should read 11/11 and chrF 100: each model memorised its
training set, and a score measured on training data is not a measurement. The two unseen lines
are the only numbers that say anything about *translation*, and they should differ. A held-out
phrase whose every word appears elsewhere can be recombined: *El perro duerme* from *perro* in
one phrase and *duerme* in another. If the model produced something else (often *El perro
corre*), sixteen passes over eleven phrases were not enough to learn that *sleeps* is *duerme*
whichever animal is doing it. A phrase with a word the model never saw cannot come out right at
all, however long you train; look at what it produced instead.

Set the two lines against your prediction before reading on.

BLEU is 0.0 on every line because a three-word sentence has no 4-grams; that is the metric's
shape, not the model's quality, which is why this notebook reports chrF and exact match.

### Also in Part B: watch a training run survive a disconnect

The A8 run takes hours, and Colab sessions drop. The `TrainingCheckpointer` every training call
above used saves at every epoch end and every ten minutes; run the same call again and it
resumes from the last save. The next cell trains a fresh model, fakes a disconnect partway
through epoch 12, then rebuilds the model from scratch and runs the same call. Read the
`[checkpoint]` lines.

In [ ]:
import tempfile


class DisconnectsAfter:
    """A stand-in for a Colab session that drops: the loader stops after this many batches."""

    def __init__(self, loader, batches):
        self.loader, self.left = loader, batches

    def __len__(self):
        return len(self.loader)

    def __iter__(self):
        for batch in self.loader:
            if self.left == 0:
                raise ConnectionError("the session dropped")
            self.left -= 1
            yield batch


DEMO_EPOCHS = 20
demo_dir = tempfile.mkdtemp(prefix="resume-demo-")   # a new, empty folder every time you run this

# 1. Start training, with a checkpointer. The "session" drops 5 batches into epoch 12.
_, loader, demo_model = build("data/train_all.tsv")
try:
    train_model(demo_model, DisconnectsAfter(loader, 11 * len(loader) + 5),
                num_epochs=DEMO_EPOCHS, device=device, config=config, gradient_clip=1.0,
                log_every=0, checkpointer=TrainingCheckpointer("resume-demo", checkpoint_dir=demo_dir))
except ConnectionError as dropped:
    print(f"\n*** {dropped}. Everything in memory is gone, including the model. ***\n")

# 2. "Reconnect": build the model again from scratch, then run the same training call.
_, loader, demo_model = build("data/train_all.tsv")
demo = train_model(demo_model, loader,
                   num_epochs=DEMO_EPOCHS, device=device, config=config, gradient_clip=1.0,
                   log_every=0, checkpointer=TrainingCheckpointer("resume-demo", checkpoint_dir=demo_dir))

print(f"\nEpochs in the loss history: {len(demo.train_losses)} of {DEMO_EPOCHS}, "
      f"though the second call trained only the last {DEMO_EPOCHS - 11}.")

**What to look at.**

- The second call printed `resumed from epoch 10` and started at epoch 12. (Its epochs count from 0, so its epoch 10 is the eleventh.)
- What was lost is the partial epoch since the last save: the five batches of epoch 12.
- In A8 the checkpoints go to Google Drive, because a Colab runtime forgets its own disk.

---

## Part C, turn in

Paste the output of the next cell into the Assignment 7 box on Learning Suite, then answer the
two questions under it in your own words. This is graded for completion and for whether the two
sentences engage with the numbers, not for the numbers.

In [ ]:
print("=== Lecture 7 toy model: report ===")
print(f"Target vocabulary V = {V};  ln(V) = {uniform_loss(V):.3f}")
print(f"Final training loss, all 12 phrases:  {losses[-1]:.3f}")
print(f"Prediction: {PREDICTION.strip() or '(none written)'}")
for r in (mine, contrast):
    print(f"Held out {r['pair'][0]!r}:  seen {r['exact_seen']}/11 exact, chrF {r['chrf_seen']:.1f};  "
          f"unseen chrF {r['chrf_unseen']:.1f}, produced {r['unseen_hyp']!r} (wanted {r['pair'][1]!r})")
print(f"BLEU on every line: {mine['bleu_seen']:.1f}")
print()
print("1. Was your prediction right? What does the difference between your two held-out phrases say about what the model learned?")
print("2. Assignment 8 trains on 100,000 pairs. Name one thing you will do differently because of this exercise.")

Lecture 8a opens with "how did the toy model go" and then "why a trained model comes out bad".
Bring your held-out result; the second slide is about it.